In [13]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [14]:
# Base URL
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

# Store scraped books
books = []

In [15]:
# Loop through all pages
for page in range(1, 51):

    url = base_url.format(page)

    response = requests.get(url)

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    book_containers = soup.find_all(
        "article",
        class_="product_pod"
    )

    print(
        f"Page {page}: {len(book_containers)} books"
    )

Page 1: 20 books
Page 2: 20 books
Page 3: 20 books
Page 4: 20 books
Page 5: 20 books
Page 6: 20 books
Page 7: 20 books
Page 8: 20 books
Page 9: 20 books
Page 10: 20 books
Page 11: 20 books
Page 12: 20 books
Page 13: 20 books
Page 14: 20 books
Page 15: 20 books
Page 16: 20 books
Page 17: 20 books
Page 18: 20 books
Page 19: 20 books
Page 20: 20 books
Page 21: 20 books
Page 22: 20 books
Page 23: 20 books
Page 24: 20 books
Page 25: 20 books
Page 26: 20 books
Page 27: 20 books
Page 28: 20 books
Page 29: 20 books
Page 30: 20 books
Page 31: 20 books
Page 32: 20 books
Page 33: 20 books
Page 34: 20 books
Page 35: 20 books
Page 36: 20 books
Page 37: 20 books
Page 38: 20 books
Page 39: 20 books
Page 40: 20 books
Page 41: 20 books
Page 42: 20 books
Page 43: 20 books
Page 44: 20 books
Page 45: 20 books
Page 46: 20 books
Page 47: 20 books
Page 48: 20 books
Page 49: 20 books
Page 50: 20 books


In [16]:
# Loop through all pages
for page in range(1, 51):

    url = base_url.format(page)

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    book_containers = soup.find_all(
        "article",
        class_="product_pod"
    )

    for book in book_containers:

        title = book.h3.a["title"]

        price = (
            book.find(
                "p",
                class_="price_color"
            )
            .text
            .replace("£", "")
        )

        availability = (
            book.find(
                "p",
                class_="instock availability"
            )
            .text
            .strip()
        )

        rating = (
            book.find("p")["class"][1]
        )

        product_url = (
            "http://books.toscrape.com/catalogue/"
            + book.h3.a["href"]
        )

        books.append(
            {
                "title": title,
                "price": price,
                "availability": availability,
                "rating": rating,
                "product_url": product_url
            }
        )

In [17]:
# Create dataframe
books_df = pd.DataFrame(books)

books_df.head()

,title,price,availability,rating,product_url
0,A Light in the Attic,Â51.77,In stock,Three,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,Â53.74,In stock,One,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,Â50.10,In stock,One,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,Â47.82,In stock,Four,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,Â54.23,In stock,Five,http://books.toscrape.com/catalogue/sapiens-a-...


In [18]:
# Check dataframe shape
books_df.shape

(1000, 5)

In [19]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         1000 non-null   object
 1   price         1000 non-null   object
 2   availability  1000 non-null   object
 3   rating        1000 non-null   object
 4   product_url   1000 non-null   object
dtypes: object(5)
memory usage: 39.2+ KB


In [20]:
# Save raw data
books_df.to_csv(
    "../data/raw/books_raw.csv",
    index=False
)

In [21]:
# Store additional information
categories = []
descriptions = []
upcs = []
taxes = []
num_available = []

for i, url in enumerate(books_df["product_url"]):

    try:

        response = requests.get(url, timeout=10)

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Category
        category = soup.find_all("a")[3].text

        # Description
        try:
            description = (
                soup.find(
                    "div",
                    id="product_description"
                )
                .find_next_sibling("p")
                .text
            )
        except:
            description = None

        # Product information table
        table = soup.find(
            "table",
            class_="table table-striped"
        )

        info = {}

        for row in table.find_all("tr"):

            key = row.th.text
            value = row.td.text

            info[key] = value

        categories.append(category)
        descriptions.append(description)
        upcs.append(info["UPC"])
        taxes.append(info["Tax"])
        num_available.append(info["Availability"])

        # Progress
        if (i+1) % 50 == 0:
            print(f"{i+1} books processed")

        # Pause briefly
        time.sleep(0.2)

    except Exception as e:

        print(f"Error on book {i+1}: {e}")

        categories.append(None)
        descriptions.append(None)
        upcs.append(None)
        taxes.append(None)
        num_available.append(None)

50 books processed
100 books processed
150 books processed
200 books processed
250 books processed
300 books processed
350 books processed
400 books processed
450 books processed
500 books processed
Error on book 511: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
550 books processed
Error on book 560: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
Error on book 598: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out. (read timeout=10)
600 books processed
Error on book 647: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
650 books processed
Error on book 656: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
Error on book 657: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
Error on book 659: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed out.
Error on book 661: HTTPConnectionPool(host='books.toscrape.com', port=80): Read timed 

In [22]:
# Add columns
books_df["category"] = categories
books_df["description"] = descriptions
books_df["upc"] = upcs
books_df["tax"] = taxes
books_df["number_available"] = num_available

In [23]:
books_df.head()

,title,price,availability,rating,product_url,category,description,upc,tax,number_available
0,A Light in the Attic,Â51.77,In stock,Three,http://books.toscrape.com/catalogue/a-light-in...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,Â£0.00,In stock (22 available)
1,Tipping the Velvet,Â53.74,In stock,One,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,Â£0.00,In stock (20 available)
2,Soumission,Â50.10,In stock,One,http://books.toscrape.com/catalogue/soumission...,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760,Â£0.00,In stock (20 available)
3,Sharp Objects,Â47.82,In stock,Four,http://books.toscrape.com/catalogue/sharp-obje...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,Â£0.00,In stock (20 available)
4,Sapiens: A Brief History of Humankind,Â54.23,In stock,Five,http://books.toscrape.com/catalogue/sapiens-a-...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,Â£0.00,In stock (20 available)


In [24]:
books_df.shape

(1000, 10)

In [25]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   title             1000 non-null   object
 1   price             1000 non-null   object
 2   availability      1000 non-null   object
 3   rating            1000 non-null   object
 4   product_url       1000 non-null   object
 5   category          992 non-null    object
 6   description       990 non-null    object
 7   upc               992 non-null    object
 8   tax               992 non-null    object
 9   number_available  992 non-null    object
dtypes: object(10)
memory usage: 78.2+ KB


In [26]:
books_df.to_csv(
    "../data/processed/books_enriched.csv",
    index=False
)